# Chapter 10 -- Graphs for Knowledge (Your Working Copy)

Work through this notebook **after reading** `notes/ch10-graphs-for-knowledge.md`. This chapter builds a small knowledge graph from 30 real short documents (extraction, a real entity-resolution bug, then dedup), implements `get_neighbors`/`find_path`/`subgraph_summary` as agent tools plus simplified local and global search, verifies the notes' 6-node PageRank dry-run in code, and compares graph-based multi-hop traversal against a naive hybrid-RAG stand-in on 10 real multi-hop questions.

Two exercises below have a stub to fill in: **bi-temporal edges** (notes Section 4) and **a query router** (notes Section 6). Everything is fully offline and deterministic -- no API key needed for either exercise.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import anthropic


def _find_and_load_env() -> None:
    """Walk up from the current working directory to find and load a repo-root .env file, if one exists."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / ".env"
        if candidate.is_file():
            load_dotenv(candidate)
            return
    print("No .env file found -- copy .env.example to .env at the repo root to enable the real-model sections.")


_find_and_load_env()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_NAME = os.getenv("BEDROCK_MODEL_ID", "anthropic.claude-sonnet-5")


def test_connection(client, model_name):
    """Send a trivial ping to confirm the Bedrock connection actually works."""
    print(f"Testing connection to Bedrock (model={model_name})...")
    try:
        response = client.messages.create(
            model=model_name, max_tokens=10,
            messages=[{"role": "user", "content": "Reply with exactly the word: pong"}],
        )
        reply = next((b.text for b in response.content if b.type == "text"), "")
        status = "PASS" if "pong" in reply.lower() else f"unexpected reply: {reply!r}"
        print(f"  {status}")
    except Exception as exc:
        print(f"  Connection check FAILED: {type(exc).__name__}: {exc}")
        print()
        print("The rest of this notebook still works fully offline -- this cell")
        print("only matters for the optional real-model section at the end.")


if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    print("AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env -- skipping connection test.")
    print("The rest of this notebook still works fully offline.")
else:
    client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    test_connection(client, MODEL_NAME)


## Part 1 -- 30 Documents, Deterministic Extraction (Given)

Notes Section 2: extraction turns unstructured text into (subject, predicate, object) facts. These 30 short documents describe two teams, four projects, and several people -- including one deliberate alias (`P. Sharma` for `Priya`) that Part 3 uses to demonstrate notes Section 8's entity-resolution problem for real.

In [ ]:
DOCUMENTS = [
    "Alice works on ProjectAtlas.",
    "Bob works on ProjectAtlas.",
    "Bob works on ProjectOrion.",
    "Carol works on ProjectNova.",
    "Dave works on ProjectComet.",
    "Dave works on ProjectNova.",
    "ProjectAtlas is owned by TeamPlatform.",
    "ProjectOrion is owned by TeamPlatform.",
    "ProjectNova is owned by TeamGrowth.",
    "ProjectComet is owned by TeamGrowth.",
    "TeamPlatform is managed by P. Sharma.",
    "TeamGrowth is managed by Omar.",
    "Priya reviews all platform team PRs.",
    "Omar runs the weekly growth sync.",
    "Alice joined the platform team in 2024.",
    "Bob previously worked on a different project.",
    "ProjectAtlas handles checkout flows.",
    "ProjectOrion is a backend migration effort.",
    "ProjectNova focuses on referral growth.",
    "ProjectComet is an experimental onboarding flow.",
    "Carol is new to the growth team.",
    "Dave splits time across two projects.",
    "The platform team ships weekly.",
    "The growth team ships bi-weekly.",
    "TeamPlatform has four engineers.",
    "TeamGrowth has three engineers.",
    "ProjectAtlas launched last quarter.",
    "ProjectComet is still in beta.",
    "Bob mentors Carol on backend patterns.",
    "Carol admires Priya's engineering leadership.",
]

print(f"{len(DOCUMENTS)} documents loaded.")
assert len(DOCUMENTS) == 30


In [ ]:
import re

WORKS_ON_RE = re.compile(r"^(\w+) works on (Project\w+)\.")
OWNED_BY_RE = re.compile(r"^(Project\w+) is owned by (Team\w+)\.")
MANAGED_BY_RE = re.compile(r"^(Team\w+) is managed by ([A-Z][\w\.]*(?:\s[A-Z][\w\.]*)?)\.")


def extract_relations(text):
    """Return a (subject, predicate, object) triple if `text` matches one of the three known patterns, else None."""
    if m := WORKS_ON_RE.match(text):
        return (m.group(1), "works_on", m.group(2))
    if m := OWNED_BY_RE.match(text):
        return (m.group(1), "owned_by", m.group(2))
    if m := MANAGED_BY_RE.match(text):
        return (m.group(1), "managed_by", m.group(2))
    return None


RAW_FACTS = [extract_relations(doc) for doc in DOCUMENTS]
RAW_FACTS = [f for f in RAW_FACTS if f is not None]

print("-" * 60)
print(f"EXTRACTION: {len(RAW_FACTS)} facts found across {len(DOCUMENTS)} documents")
print("-" * 60)
for subject, predicate, object_ in RAW_FACTS:
    print(f"  ({subject}, {predicate}, {object_})")


## Part 2 -- The Graph, and a Real Entity-Resolution Bug

`build_graph` turns the raw facts into an adjacency structure -- **before** any deduplication. The source documents recorded the platform team's manager under her formal name, `P. Sharma`; a separate, non-structural sentence mentions her informally as `Priya`, but that mention never became a graph edge. Querying by the name a person would actually use finds nothing, even though she's fully present in the graph under a different string -- notes Section 8's entity-resolution problem, seen here as a missing expected name rather than a duplicate.

In [ ]:
from collections import defaultdict


def build_graph(facts):
    """Directed multigraph: graph[subject] = [(predicate, object), ...]."""
    graph = defaultdict(list)
    for subject, predicate, object_ in facts:
        graph[subject].append((predicate, object_))
    return graph


def get_neighbors(graph, entity):
    """notes Section 7's first traversal tool -- direct outgoing edges only."""
    return graph.get(entity, [])


def find_path(graph, start, end, max_hops=4):
    """BFS shortest path -- notes Section 7's find_path tool. Returns a list of (from, relation, to) or None."""
    if start == end:
        return []
    frontier = [(start, [])]
    visited = {start}
    while frontier:
        node, path = frontier.pop(0)
        if len(path) >= max_hops:
            continue
        for predicate, neighbor in get_neighbors(graph, node):
            if neighbor == end:
                return path + [(node, predicate, neighbor)]
            if neighbor not in visited:
                visited.add(neighbor)
                frontier.append((neighbor, path + [(node, predicate, neighbor)]))
    return None


graph_before_dedup = build_graph(RAW_FACTS)

print("Neighbors of ProjectAtlas (before dedup):", get_neighbors(graph_before_dedup, "ProjectAtlas"))
print("Neighbors of TeamPlatform (before dedup):", get_neighbors(graph_before_dedup, "TeamPlatform"))
print()
path_to_sharma = find_path(graph_before_dedup, "Alice", "P. Sharma")
path_to_priya = find_path(graph_before_dedup, "Alice", "Priya")
print(f"find_path(Alice, P. Sharma) = {path_to_sharma}")
print(f"find_path(Alice, Priya)     = {path_to_priya}")

assert path_to_sharma is not None, "a path to P. Sharma (the recorded structural name) should exist"
assert path_to_priya is None, "BEFORE dedup, querying by the informal name 'Priya' should find nothing -- it was never recorded as a graph edge"
print()
print("Confirmed: the real path from Alice to her team's manager exists in the graph --")
print("but only reachable under the formal name 'P. Sharma'. Querying by the name")
print("everyone actually uses, 'Priya', silently finds nothing. Notes Section 8, exactly.")


In [ ]:
ENTITY_ALIASES = {"P. Sharma": "Priya"}


def resolve_entities(facts, aliases):
    """Canonicalize subject/object names through the alias table before graph construction."""
    resolved = []
    for subject, predicate, object_ in facts:
        subject = aliases.get(subject, subject)
        object_ = aliases.get(object_, object_)
        resolved.append((subject, predicate, object_))
    return resolved


RESOLVED_FACTS = resolve_entities(RAW_FACTS, ENTITY_ALIASES)
graph = build_graph(RESOLVED_FACTS)

print("Neighbors of TeamPlatform (after dedup):", get_neighbors(graph, "TeamPlatform"))
path_to_priya_after = find_path(graph, "Alice", "Priya")
print(f"\nfind_path(Alice, Priya) after dedup = {path_to_priya_after}")

managed_by_edges = [(s, p, o) for s, p, o in RESOLVED_FACTS if s == "TeamPlatform" and p == "managed_by"]
assert path_to_priya_after is not None, "after dedup, querying by the informal name 'Priya' should now find the real path"
assert managed_by_edges == [("TeamPlatform", "managed_by", "Priya")]
print(f"\nTeamPlatform's managed_by edge now points at the canonical name: {managed_by_edges}")
print("Resolving the alias BEFORE graph construction is what makes the natural query")
print("(asking about 'Priya') find the answer at all.")


## Part 3 -- PageRank, Verified Against Notes Section 11

The exact 6-node subgraph from the notes dry-run, computed here in code.

In [ ]:
def pagerank(graph, nodes, damping=0.85, iterations=2):
    """Standard PageRank recurrence -- notes Section 11. Dangling nodes are left unredistributed, as in the notes."""
    n = len(nodes)
    base = (1 - damping) / n
    out_degree = {node: len(get_neighbors(graph, node)) for node in nodes}
    in_links = {node: [] for node in nodes}
    for node in nodes:
        for predicate, target in get_neighbors(graph, node):
            if target in in_links:
                in_links[target].append(node)

    scores = {node: 1 / n for node in nodes}
    history = []
    for _ in range(iterations):
        new_scores = {}
        for node in nodes:
            inbound = sum(scores[m] / out_degree[m] for m in in_links[node] if out_degree[m] > 0)
            new_scores[node] = base + damping * inbound
        scores = new_scores
        history.append(dict(scores))
    return history


DRYRUN_NODES = ["Alice", "Bob", "ProjectAtlas", "ProjectOrion", "TeamPlatform", "Priya"]
dryrun_facts = [
    ("Alice", "works_on", "ProjectAtlas"), ("Bob", "works_on", "ProjectAtlas"), ("Bob", "works_on", "ProjectOrion"),
    ("ProjectAtlas", "owned_by", "TeamPlatform"), ("ProjectOrion", "owned_by", "TeamPlatform"),
    ("TeamPlatform", "managed_by", "Priya"),
]
dryrun_graph = build_graph(dryrun_facts)
history = pagerank(dryrun_graph, DRYRUN_NODES)

for i, scores in enumerate(history, start=1):
    print(f"Iteration {i}: " + "  ".join(f"{node}={scores[node]:.4f}" for node in DRYRUN_NODES))

final = history[-1]
ranked = sorted(DRYRUN_NODES, key=lambda n: -final[n])
print(f"\nFinal ranking: {ranked}")

assert abs(final["TeamPlatform"] - 0.3083) < 0.001
assert abs(final["Priya"] - 0.2871) < 0.001
assert ranked[0] == "TeamPlatform" and ranked[1] == "Priya"
print("\nMatches notes Section 11 exactly: TeamPlatform and Priya top the ranking through")
print("structural convergence, not raw edge count.")


## Part 4 -- Local Search, Global Search, and `subgraph_summary` (Given)

`local_search` walks outward from a specific entity (notes Section 3's local mode). `global_search` is a deliberately simplified stand-in for real Leiden community detection plus LLM summarization -- it groups entities by team and returns a canned summary per group, illustrating the *local vs global* distinction concretely without the full pipeline's cost.

In [ ]:
def subgraph_summary(graph, entity, hops=2):
    """Notes Section 7's third tool -- a bounded neighborhood, not the whole graph."""
    visited = {entity}
    frontier = [entity]
    edges = []
    for _ in range(hops):
        next_frontier = []
        for node in frontier:
            for predicate, neighbor in get_neighbors(graph, node):
                edges.append((node, predicate, neighbor))
                if neighbor not in visited:
                    visited.add(neighbor)
                    next_frontier.append(neighbor)
        frontier = next_frontier
    return {"entities": sorted(visited), "edges": edges}


def local_search(graph, entity, hops=2):
    """Start from a specific entity and walk outward -- notes Section 3's local mode."""
    return subgraph_summary(graph, entity, hops=hops)


TEAM_MEMBERSHIP = {
    "TeamPlatform": ["Alice", "Bob", "Priya", "ProjectAtlas", "ProjectOrion"],
    "TeamGrowth": ["Carol", "Dave", "Omar", "ProjectNova", "ProjectComet"],
}


def global_search(query_keyword=None):
    """
    A simplified stand-in for Leiden community detection + hierarchical
    summarization (notes Section 3) -- pre-written per-team summaries,
    grouped and returned without re-reading source documents.
    """
    summaries = {
        "TeamPlatform": "TeamPlatform (managed by Priya) owns ProjectAtlas and ProjectOrion; Alice and Bob work on this team's projects.",
        "TeamGrowth": "TeamGrowth (managed by Omar) owns ProjectNova and ProjectComet; Carol and Dave work on this team's projects.",
    }
    if query_keyword:
        return {team: s for team, s in summaries.items() if query_keyword.lower() in s.lower() or query_keyword.lower() in team.lower()}
    return summaries


print("local_search(Alice, hops=2):")
print(" ", local_search(graph, "Alice"))
print()
print("global_search() -- pre-written community summaries, no source re-read:")
for team, summary in global_search().items():
    print(f"  {team}: {summary}")


## Part 5 -- 10 Multi-Hop Questions: Graph Traversal vs a Naive Hybrid-RAG Stand-In

`hybrid_rag_answer` is a deliberately simple stand-in for a keyword/vector hybrid pipeline (not the full B03 implementation) -- it scores every document by word overlap with the question and returns the single top-scoring document's text as its answer. `graph_answer` runs the actual 2-hop or 3-hop traversal notes Section 1 describes. The comparison is the point: multi-hop information is split across documents that never co-occur, so a single best-matching document structurally cannot carry both facts at once.

In [ ]:
def hybrid_rag_answer(question, documents):
    """Naive keyword-overlap retrieval -- returns the single most-similar document's text."""
    q_words = set(question.lower().replace('?', '').replace('.', '').split())
    scored = []
    for doc in documents:
        d_words = set(doc.lower().replace('.', '').split())
        scored.append((len(q_words & d_words), doc))
    scored.sort(key=lambda t: -t[0])
    return scored[0][1]


PERSON_RE = re.compile(r"(?:project\(s\) )?(Alice|Bob|Carol|Dave)")
TEAM_RE = re.compile(r"(TeamPlatform|TeamGrowth)")


def graph_answer(question, graph):
    """A small, deterministic 'agent' that runs the right traversal for one of three fixed question templates."""
    lowered = question.lower()
    if "manages the team" in lowered:
        person = PERSON_RE.search(question).group(1)
        project = get_neighbors(graph, person)[0][1]
        team = [o for p, o in get_neighbors(graph, project) if p == "owned_by"][0]
        manager = [o for p, o in get_neighbors(graph, team) if p == "managed_by"]
        return manager[0]
    if "which team owns" in lowered:
        person = PERSON_RE.search(question).group(1)
        project = get_neighbors(graph, person)[0][1]
        team = [o for p, o in get_neighbors(graph, project) if p == "owned_by"][0]
        return team
    if "name someone who works on" in lowered:
        team = TEAM_RE.search(question).group(1)
        projects = [s for s, p, o in RESOLVED_FACTS if p == "owned_by" and o == team]
        people = [s for s, p, o in RESOLVED_FACTS if p == "works_on" and o in projects]
        return people[0]
    return None


QUESTIONS = [
    ("Which team owns the project Alice works on?", "TeamPlatform"),
    ("Which team owns the project Bob works on?", "TeamPlatform"),
    ("Which team owns the project Carol works on?", "TeamGrowth"),
    ("Which team owns the project Dave works on?", "TeamGrowth"),
    ("Who manages the team that owns the project Alice works on?", "Priya"),
    ("Who manages the team that owns the project Bob works on?", "Priya"),
    ("Who manages the team that owns the project Carol works on?", "Omar"),
    ("Who manages the team that owns the project Dave works on?", "Omar"),
    ("Name someone who works on a project owned by TeamPlatform.", "Alice"),
    ("Name someone who works on a project owned by TeamGrowth.", "Carol"),
]
assert len(QUESTIONS) == 10

graph_correct = 0
rag_correct = 0
print("-" * 60)
print(f"{'question':<58} {'graph':<14} {'hybrid-RAG contains answer?'}")
print("-" * 60)
for question, expected in QUESTIONS:
    graph_result = graph_answer(question, graph)
    rag_doc = hybrid_rag_answer(question, DOCUMENTS)
    rag_hit = expected in rag_doc
    graph_hit = graph_result == expected
    graph_correct += graph_hit
    rag_correct += rag_hit
    print(f"{question:<58} {str(graph_result):<14} {'YES' if rag_hit else 'no'}  (top doc: {rag_doc!r})")

print(f"\nGraph traversal: {graph_correct}/10 correct.")
print(f"Hybrid-RAG stand-in (single top document contains the answer): {rag_correct}/10.")
assert graph_correct == 10, "graph traversal should get every one of these multi-hop questions right"
assert rag_correct <= 2, "the naive single-document hybrid-RAG stand-in should fail on almost all of these by construction"
print("\nConfirmed: multi-hop information is split across documents that never co-occur,")
print("so no single best-matching document can carry both facts -- notes Section 1, measured.")


## Exercise 1 -- Bi-Temporal Edges

Implement `add_temporal_edge(store, subject, predicate, object_, valid_at, created_at)` and `invalidate_edge(store, subject, predicate, object_, invalid_at, expired_at)`: notes Section 4's two independent clocks. `store` is a plain list of edge dicts, each with `subject`, `predicate`, `object`, `valid_at`, `invalid_at` (`None` until invalidated), `created_at`, `expired_at` (`None` until invalidated). Invalidating an edge must **set its timestamps, never remove it from the list** -- history stays queryable.

In [ ]:
def add_temporal_edge(store, subject, predicate, object_, valid_at, created_at):
    """Append a new bi-temporal edge, active (not yet invalidated)."""
    # TODO: append a dict to `store` with keys subject, predicate, object,
    # valid_at, invalid_at (None), created_at, expired_at (None).
    pass


def invalidate_edge(store, subject, predicate, object_, invalid_at, expired_at):
    """Set invalid_at/expired_at on the matching ACTIVE edge -- never delete it."""
    # TODO: find the edge in `store` matching (subject, predicate, object)
    # that is still active (invalid_at is None), and set its invalid_at and
    # expired_at fields. Do NOT remove it from the list.
    pass


def active_edges(store, as_of=None):
    """Edges with no invalid_at, or (for historical queries) valid at a given point in time."""
    if as_of is None:
        return [e for e in store if e["invalid_at"] is None]
    return [e for e in store if e["valid_at"] <= as_of and (e["invalid_at"] is None or e["invalid_at"] > as_of)]


In [ ]:
temporal_store = []
add_temporal_edge(temporal_store, "Bob", "leads", "ProjectAtlas", valid_at=1, created_at=1)

# Later, leadership changes -- Alice takes over.
invalidate_edge(temporal_store, "Bob", "leads", "ProjectAtlas", invalid_at=10, expired_at=10)
add_temporal_edge(temporal_store, "Alice", "leads", "ProjectAtlas", valid_at=10, created_at=10)

print(f"Total edges in store (nothing deleted): {len(temporal_store)}")
for e in temporal_store:
    print(f"  {e}")

current = active_edges(temporal_store)
historical_at_5 = active_edges(temporal_store, as_of=5)

assert len(temporal_store) == 2, "invalidating must NOT remove the old edge from the store"
assert [e['subject'] for e in current] == ["Alice"], "the currently active leader should be Alice"
assert [e['subject'] for e in historical_at_5] == ["Bob"], "as of time=5, Bob was still the valid leader"

print(f"\nCurrently active: {[e['subject'] for e in current]}")
print(f"As of time=5 (historical query): {[e['subject'] for e in historical_at_5]}")
print("\nExercise 1 PASSED -- both edges persist in the store; 'current' and 'as-of-time-5'")
print("queries correctly return different answers from the SAME underlying data, exactly")
print("notes Section 4's point about reconstructing what was believed at any past point.")


## Exercise 2 -- A Query Router

Implement `classify_query(question)`: notes Section 6's router, dispatching each question to one of `"vector"`, `"graph"`, `"sql"`, or `"keyword"` by inspecting its wording. Use these cues, checked in order: if the question contains "how many", "average", "total", or "count" -> `"sql"`. If it contains "which team", "who manages", "connected to", or "owns" -> `"graph"`. If it looks like a lookup for an exact code/ID (`"error code"`, `"ticket #"`, or a token containing a digit) -> `"keyword"`. Otherwise -> `"vector"`.

In [ ]:
def classify_query(question):
    """Route a question to vector / graph / sql / keyword by its wording -- notes Section 6."""
    # TODO: lowercase `question`. Return "sql" if it contains "how many",
    # "average", "total", or "count". Return "graph" if it contains
    # "which team", "who manages", "connected to", or "owns". Return
    # "keyword" if it contains "error code", "ticket #", or the ORIGINAL
    # (non-lowered) question has any digit character. Otherwise return
    # "vector".
    return "vector"


In [ ]:
ROUTER_TEST_CASES = [
    ("Which team owns the project Alice works on?", "graph"),
    ("How many engineers are on TeamPlatform?", "sql"),
    ("What's the average ticket resolution time this quarter?", "sql"),
    ("What is error code E4471?", "keyword"),
    ("Find documents about onboarding flows in general.", "vector"),
    ("Who manages the team that owns ProjectComet?", "graph"),
    ("What's ticket #8823 about?", "keyword"),
    ("Summarize the themes across our engineering docs.", "vector"),
]

correct = 0
print("-" * 60)
print("QUERY ROUTER TEST")
print("-" * 60)
for question, expected in ROUTER_TEST_CASES:
    predicted = classify_query(question)
    is_correct = predicted == expected
    correct += is_correct
    print(f"  {'OK ' if is_correct else 'ERR'} expected={expected:8s} predicted={predicted:8s} {question!r}")

accuracy = correct / len(ROUTER_TEST_CASES)
print(f"\nAccuracy: {correct}/{len(ROUTER_TEST_CASES)} = {accuracy:.0%}")
assert accuracy == 1.0, f"expected the router to classify all test cases correctly, got {accuracy:.0%}"
print("\nExercise 2 PASSED -- each query shape (relationship, aggregate, exact-ID, fuzzy)")
print("routes to the matching retrieval mode, notes Section 6's whole argument.")


## Optional -- Have a Real Claude Model Extract Relations From a Document

The same extraction job Part 1 did with regexes, asked of a real model instead -- useful specifically because a real model can catch relation types the fixed regex patterns above were never written to look for.

In [ ]:
RUN_REAL_EXTRACTION_DEMO = False


def run_real_extraction_demo():
    if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
        print("Skipping real extraction demo: AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env.")
        return

    real_client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    prompt = (
        "Extract every (subject, relation, object) triple you can find in this sentence: "
        "\"Bob mentors Carol on backend patterns.\" "
        "Our fixed regex patterns only know about works_on/owned_by/manages -- what does a real model find?"
    )
    try:
        response = real_client.messages.create(
            model=MODEL_NAME, max_tokens=150,
            messages=[{"role": "user", "content": prompt}],
        )
        text = next((b.text for b in response.content if b.type == "text"), "")
        print(text)
    except Exception as exc:
        print(f"Real extraction demo failed: {type(exc).__name__}: {exc}")


if RUN_REAL_EXTRACTION_DEMO:
    run_real_extraction_demo()
else:
    print("RUN_REAL_EXTRACTION_DEMO is False -- running in offline/regex mode only.")
    print("Flip it to True to compare against a real Claude model's open extraction via Bedrock.")


## Key Takeaways

You built a real graph from 30 documents, watched a genuine entity-resolution bug break a query for an alias before fixing it with a resolution table, and verified the notes' PageRank dry-run exactly in code. The 10-question comparison made notes Section 1's abstract claim measurable: graph traversal got all 10 multi-hop questions right, while a naive single-document retrieval stand-in got at most 2, by construction, because the connecting facts never co-occur in one document. Bi-temporal edges showed the same underlying data correctly answering two different questions -- "what's true now" and "what was true at time=5" -- something a single-timestamp supersession model (Chapter 9) can't do. The query router closed the loop on notes Section 6: match the retrieval mechanism to the question's actual shape, not a single pipeline for everything.

**Connection forward:** Chapter 11 leaves knowledge representation behind for execution representation -- the graph/state-machine runtime that controls an agent's own control flow. Several ideas carry over structurally (nodes, edges, traversal) even though what they represent changes completely.